# 1.PIIMiddleware中间件
## 举例1：使用内置检测器

In [ ]:
import os

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, PIIMiddleware
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from scripts.regsetup import description
from rich import print as rprint
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    profile={
        "max_input_tokens": 1_000_000
    },
    # 关键修改：关闭思考模式
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    },
)

In [ ]:
from langchain.agents.middleware import PIIMiddleware
from langchain.chat_models import init_chat_model
agent=create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email",strategy="redact",apply_to_input=True),
        PIIMiddleware("credit_card",strategy="mask",apply_to_input=True),
        PIIMiddleware("url",strategy="hash",apply_to_input=True),
        PIIMiddleware("mac_address",strategy="mask",apply_to_input=True),
        PIIMiddleware("ip",strategy="block",apply_to_input=True),
    ]
)
response=agent.invoke({
    "messages":[HumanMessage("""
    帮我向 user@example.invalid 发送一封邮件
    同时查看银行卡号： 5105-1051-0510-5100 的余额
    访问 https://localhost:12345
    确认这是不是 MAC地址： 11-11-11-11-11-11
    """)]
})
for msg in response["messages"]:
    msg.pretty_print()

In [ ]:
try:
    response1 = agent.invoke({
    "messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]
})
except Exception as e:
    print(f"检测到ip，抛出异常{e}")

## 举例2：自定义检测器/函数

In [ ]:
import re
# 自定义检测函数
def detect_phone_number(content: str):
    return [
    {
        "text": m.group(0), # 提取出具体匹配到的 11 位数字文本（例如00000000000）
        "start": m.start(), # 这段数字在原文本中的“起始索引位置”（从 0 开始算）
        "end": m.end() # 这段数字在原文本中的“结束索引位置”
    } for m in re.finditer(r"[0-9]{11}", content)
]

In [ ]:
text = "用户甲的电话是00000000000，用户乙的电话是11111111111。"
result = detect_phone_number(text)
print(result)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage
agent = create_agent(
            model=model,
            tools=[],
            middleware=[
            PIIMiddleware("api_key", strategy="hash", apply_to_input=True,
            detector=r"sk-[a-zA-Z0-9]+"),
            PIIMiddleware("phone_number", strategy="mask", apply_to_input=True,detector=detect_phone_number)
            ])
response = agent.invoke({
            "messages": [HumanMessage("""
            这是不是有效的 API_KEY： <虚构测试值，仅用于脱敏演示>
            帮我给这个号码打电话： 00000000000
            访问 https://localhost:12345
             """)]
})
for msg in response["messages"]:
    msg.pretty_print()